# Exercise 5 — MarketDataStore

`MarketDataStore` ties everything together: it fetches, validates, stores, and serves OHLCV data for multiple tickers through a clean four-method interface. The `fetch_fn` injection keeps the gate deterministic — pass `_synthetic` to avoid live API calls.

In [ ]:
import pandas as pd, math, sqlite3, tempfile, os

OHLCV_COLS = ["Open", "High", "Low", "Close", "Volume"]

def _synthetic(ticker="TEST", period="1y", interval="1d", n=50):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def validate_ohlcv(df):
    if not isinstance(df, pd.DataFrame): return False, "not a DataFrame"
    missing = [c for c in OHLCV_COLS if c not in df.columns]
    if missing: return False, "missing columns: " + ", ".join(missing)
    if len(df) == 0: return False, "DataFrame is empty"
    valid = df.dropna(subset=["High", "Low"])
    if len(valid) > 0 and (valid["High"] < valid["Low"]).any():
        return False, "High < Low detected"
    return True, ""
def normalize_ohlcv(df):
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex): df.index = pd.to_datetime(df.index)
    if df.index.tz is not None: df.index = df.index.tz_localize(None)
    return df[[c for c in OHLCV_COLS if c in df.columns]]
def fetch_ohlcv(ticker, period="1y", interval="1d", fetch_fn=None):
    if fetch_fn is not None: return fetch_fn(ticker, period, interval)
    import yfinance as yf
    df = yf.download(ticker, period=period, interval=interval, progress=False)
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
    return df
def store_ohlcv(df, ticker, db_path=":memory:"):
    ok, reason = validate_ohlcv(df)
    if not ok: raise ValueError("Invalid OHLCV: " + reason)
    df = normalize_ohlcv(df)
    con = sqlite3.connect(db_path)
    try:
        con.execute("CREATE TABLE IF NOT EXISTS ohlcv (ticker TEXT NOT NULL, date TEXT NOT NULL, open REAL, high REAL, low REAL, close REAL, volume REAL, PRIMARY KEY (ticker, date))")
        rows = []
        for date, row in df.iterrows():
            d = str(date.date()) if hasattr(date, "date") else str(date)
            rows.append((ticker, d, float(row["Open"]), float(row["High"]),
                         float(row["Low"]), float(row["Close"]), float(row["Volume"])))
        con.executemany("INSERT OR REPLACE INTO ohlcv VALUES (?,?,?,?,?,?,?)", rows)
        con.commit(); return len(rows)
    finally: con.close()

def load_ohlcv(ticker, db_path=":memory:"):
    con = sqlite3.connect(db_path)
    try:
        try:
            rows = con.execute(
                "SELECT date, open, high, low, close, volume FROM ohlcv "
                "WHERE ticker=? ORDER BY date", (ticker,)).fetchall()
        except sqlite3.OperationalError:
            return pd.DataFrame(columns=OHLCV_COLS)
        if not rows: return pd.DataFrame(columns=OHLCV_COLS)
        df = pd.DataFrame(rows, columns=["date", "Open", "High", "Low", "Close", "Volume"])
        df.index = pd.to_datetime(df["date"]); df.index.name = None
        return df.drop(columns=["date"])
    finally: con.close()

# ── Exercise: implement MarketDataStore ──────────────────────────────────────

class MarketDataStore:
    """Fetch, validate, and persist OHLCV data for multiple tickers.

    Args:
        db_path  : SQLite file path.
        fetch_fn : callable(ticker, period, interval) -> DataFrame, or None.

    Methods:
        fetch(ticker, period, interval)   -> pd.DataFrame  raw, no store
        update(ticker, period, interval)  -> int           fetch + store
        load(ticker)                      -> pd.DataFrame  from SQLite
        tickers()                         -> list[str]     sorted, seen
    """

    def __init__(self, db_path=":memory:", fetch_fn=None):
        # TODO: store db_path, fetch_fn, and a set for seen tickers
        self._db       = db_path
        self._fetch_fn = fetch_fn
        self._tickers  = set()

    def fetch(self, ticker, period="1y", interval="1d"):
        # TODO: call fetch_ohlcv(ticker, period, interval, fetch_fn=self._fetch_fn)
        return fetch_ohlcv(ticker, period=period, interval=interval, fetch_fn=self._fetch_fn)

    def update(self, ticker, period="1y", interval="1d"):
        # TODO: fetch + store_ohlcv(df, ticker, self._db) + add ticker to set + return n
        return 0

    def load(self, ticker):
        # TODO: return load_ohlcv(ticker, self._db)
        return pd.DataFrame(columns=OHLCV_COLS)

    def tickers(self):
        # TODO: return sorted(self._tickers)
        return []


### Checks

In [ ]:
checks = 0

# 1 — update() fetches and stores data, returns row count
try:
    with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as f:
        _db1 = f.name
    try:
        store = MarketDataStore(db_path=_db1, fetch_fn=_synthetic)
        n = store.update("AAPL")
        assert n == 50, f"expected 50, got {n}"
        checks += 1; print("✅ 1 update() stores data and returns row count")
    finally:
        os.unlink(_db1)
except Exception as e:
    print("❌ 1:", e)

# 2 — load() returns stored data with DatetimeIndex
try:
    with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as f:
        _db2 = f.name
    try:
        store = MarketDataStore(db_path=_db2, fetch_fn=_synthetic)
        store.update("AAPL")
        df = store.load("AAPL")
        assert len(df) == 50
        assert isinstance(df.index, pd.DatetimeIndex)
        assert "Close" in df.columns
        checks += 1; print("✅ 2 load() returns stored data with DatetimeIndex")
    finally:
        os.unlink(_db2)
except Exception as e:
    print("❌ 2:", e)

# 3 — tickers() returns sorted list of updated tickers
try:
    with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as f:
        _db3 = f.name
    try:
        store = MarketDataStore(db_path=_db3, fetch_fn=_synthetic)
        store.update("MSFT"); store.update("AAPL"); store.update("GOOG")
        t = store.tickers()
        assert t == sorted(t), f"not sorted: {t}"
        assert "AAPL" in t and "MSFT" in t and "GOOG" in t
        checks += 1; print("✅ 3 tickers() returns sorted list of updated tickers")
    finally:
        os.unlink(_db3)
except Exception as e:
    print("❌ 3:", e)

# 4 — load() for missing ticker returns empty DataFrame
try:
    with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as f:
        _db4 = f.name
    try:
        store = MarketDataStore(db_path=_db4, fetch_fn=_synthetic)
        empty = store.load("NOTEXIST")
        assert isinstance(empty, pd.DataFrame) and len(empty) == 0
        checks += 1; print("✅ 4 load() for missing ticker returns empty DataFrame")
    finally:
        os.unlink(_db4)
except Exception as e:
    print("❌ 4:", e)

# 5 — fetch() uses the injected fetch_fn and does NOT store
try:
    calls = []
    def _tracking(ticker, period, interval):
        calls.append(ticker); return _synthetic()
    with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as f:
        _db5 = f.name
    try:
        store = MarketDataStore(db_path=_db5, fetch_fn=_tracking)
        df = store.fetch("AAPL")
        assert "AAPL" in calls
        still_empty = store.load("AAPL")
        assert len(still_empty) == 0, "fetch() should not store data"
        checks += 1; print("✅ 5 fetch() uses fetch_fn and does not persist data")
    finally:
        os.unlink(_db5)
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
